# Part 2: Train Transformer Models
## BERT, RoBERTa, and XLM-RoBERTa for Sentiment Analysis

In this notebook, we'll:
1. Load preprocessed data
2. Train BERT, RoBERTa, and XLM-RoBERTa models
3. Evaluate and compare performance
4. Generate visualizations

---

## Setup

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizer, BertForSequenceClassification,
    RobertaTokenizer, RobertaForSequenceClassification,
    XLMRobertaTokenizer, XLMRobertaForSequenceClassification,
    AdamW, get_linear_schedule_with_warmup
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Load Preprocessed Data

In [ ]:
# Load preprocessed data from Part 1
df = pd.read_csv('preprocessed_reviews.csv')
print(f"✓ Loaded {len(df):,} reviews")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSentiment distribution:")
print(df['sentiment'].value_counts())

## Create PyTorch Dataset

In [ ]:
class ReviewDataset(Dataset):
    """PyTorch Dataset for restaurant reviews."""
    
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

print("✓ Dataset class defined")

## Prepare Data Splits

In [ ]:
from sklearn.model_selection import train_test_split

# Create label mapping
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['sentiment'].map(label_map)

# Split data
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

## Training Configuration

In [ ]:
# ⚠️ TRAINING CONFIGURATION - Adjust these parameters
CONFIG = {
    'batch_size': 16,        # Reduce to 8 or 4 if out of memory
    'epochs': 3,             # Increase to 4-5 for better results
    'learning_rate': 2e-5,   # Standard for transformers
    'max_length': 128,       # Maximum sequence length
    'text_column': 'cleaned_text',  # Column with text data
}

print("Training Configuration:")
print("="*50)
for key, value in CONFIG.items():
    print(f"  {key:20}: {value}")
print("\n💡 Tip: Reduce batch_size if you get out-of-memory errors")

## Helper Functions

In [ ]:
def train_epoch(model, data_loader, optimizer, scheduler, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(data_loader, desc='Training')
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        total_loss += loss.item()
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        progress_bar.set_postfix({'loss': loss.item()})
    
    return total_loss / len(data_loader)

def evaluate(model, data_loader, device):
    """Evaluate model."""
    model.eval()
    total_loss = 0
    predictions = []
    true_labels = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            
            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            
            predictions.extend(preds.cpu().tolist())
            true_labels.extend(labels.cpu().tolist())
    
    avg_loss = total_loss / len(data_loader)
    accuracy = accuracy_score(true_labels, predictions)
    
    return avg_loss, accuracy, predictions, true_labels

print("✓ Helper functions defined")

## Model 1: BERT (English)

Training BERT base model for sentiment classification.

In [ ]:
print("="*70)
print("TRAINING BERT MODEL")
print("="*70)

# Initialize BERT
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)
bert_model.to(device)

# Create datasets
train_dataset = ReviewDataset(
    train_df[CONFIG['text_column']].tolist(),
    train_df['label'].tolist(),
    bert_tokenizer,
    CONFIG['max_length']
)
val_dataset = ReviewDataset(
    val_df[CONFIG['text_column']].tolist(),
    val_df['label'].tolist(),
    bert_tokenizer,
    CONFIG['max_length']
)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'])

# Optimizer and scheduler
optimizer = AdamW(bert_model.parameters(), lr=CONFIG['learning_rate'])
total_steps = len(train_loader) * CONFIG['epochs']
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

print(f"✓ BERT initialized")
print(f"  Training samples: {len(train_dataset):,}")
print(f"  Validation samples: {len(val_dataset):,}")
print(f"  Batches per epoch: {len(train_loader)}")

In [ ]:
# Train BERT
bert_history = {'train_loss': [], 'val_loss': [], 'val_accuracy': []}

for epoch in range(CONFIG['epochs']):
    print(f"\nEpoch {epoch + 1}/{CONFIG['epochs']}")
    print("-" * 50)
    
    # Train
    train_loss = train_epoch(bert_model, train_loader, optimizer, scheduler, device)
    
    # Validate
    val_loss, val_accuracy, _, _ = evaluate(bert_model, val_loader, device)
    
    # Save history
    bert_history['train_loss'].append(train_loss)
    bert_history['val_loss'].append(val_loss)
    bert_history['val_accuracy'].append(val_accuracy)
    
    print(f"\nEpoch {epoch + 1} Results:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")
    print(f"  Val Acc:    {val_accuracy:.4f}")

print("\n✓ BERT training completed!")

## Model 2: RoBERTa (English)

Training RoBERTa model - optimized version of BERT.

In [ ]:
print("="*70)
print("TRAINING RoBERTa MODEL")
print("="*70)

# Initialize RoBERTa
roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
roberta_model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=3)
roberta_model.to(device)

# Create datasets
train_dataset_roberta = ReviewDataset(
    train_df[CONFIG['text_column']].tolist(),
    train_df['label'].tolist(),
    roberta_tokenizer,
    CONFIG['max_length']
)
val_dataset_roberta = ReviewDataset(
    val_df[CONFIG['text_column']].tolist(),
    val_df['label'].tolist(),
    roberta_tokenizer,
    CONFIG['max_length']
)

# Create data loaders
train_loader_roberta = DataLoader(train_dataset_roberta, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader_roberta = DataLoader(val_dataset_roberta, batch_size=CONFIG['batch_size'])

# Optimizer and scheduler
optimizer_roberta = AdamW(roberta_model.parameters(), lr=CONFIG['learning_rate'])
scheduler_roberta = get_linear_schedule_with_warmup(optimizer_roberta, num_warmup_steps=0, num_training_steps=total_steps)

print(f"✓ RoBERTa initialized")

In [ ]:
# Train RoBERTa
roberta_history = {'train_loss': [], 'val_loss': [], 'val_accuracy': []}

for epoch in range(CONFIG['epochs']):
    print(f"\nEpoch {epoch + 1}/{CONFIG['epochs']}")
    print("-" * 50)
    
    # Train
    train_loss = train_epoch(roberta_model, train_loader_roberta, optimizer_roberta, scheduler_roberta, device)
    
    # Validate
    val_loss, val_accuracy, _, _ = evaluate(roberta_model, val_loader_roberta, device)
    
    # Save history
    roberta_history['train_loss'].append(train_loss)
    roberta_history['val_loss'].append(val_loss)
    roberta_history['val_accuracy'].append(val_accuracy)
    
    print(f"\nEpoch {epoch + 1} Results:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")
    print(f"  Val Acc:    {val_accuracy:.4f}")

print("\n✓ RoBERTa training completed!")

## Model 3: XLM-RoBERTa (Multilingual)

Training XLM-RoBERTa - the best model for multilingual tasks!

In [ ]:
print("="*70)
print("TRAINING XLM-RoBERTa MODEL (MULTILINGUAL)")
print("="*70)

# Initialize XLM-RoBERTa
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
xlm_model = XLMRobertaForSequenceClassification.from_pretrained('xlm-roberta-base', num_labels=3)
xlm_model.to(device)

# Create datasets
train_dataset_xlm = ReviewDataset(
    train_df[CONFIG['text_column']].tolist(),
    train_df['label'].tolist(),
    xlm_tokenizer,
    CONFIG['max_length']
)
val_dataset_xlm = ReviewDataset(
    val_df[CONFIG['text_column']].tolist(),
    val_df['label'].tolist(),
    xlm_tokenizer,
    CONFIG['max_length']
)

# Create data loaders
train_loader_xlm = DataLoader(train_dataset_xlm, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader_xlm = DataLoader(val_dataset_xlm, batch_size=CONFIG['batch_size'])

# Optimizer and scheduler
optimizer_xlm = AdamW(xlm_model.parameters(), lr=CONFIG['learning_rate'])
scheduler_xlm = get_linear_schedule_with_warmup(optimizer_xlm, num_warmup_steps=0, num_training_steps=total_steps)

print(f"✓ XLM-RoBERTa initialized")

In [ ]:
# Train XLM-RoBERTa
xlm_history = {'train_loss': [], 'val_loss': [], 'val_accuracy': []}

for epoch in range(CONFIG['epochs']):
    print(f"\nEpoch {epoch + 1}/{CONFIG['epochs']}")
    print("-" * 50)
    
    # Train
    train_loss = train_epoch(xlm_model, train_loader_xlm, optimizer_xlm, scheduler_xlm, device)
    
    # Validate
    val_loss, val_accuracy, _, _ = evaluate(xlm_model, val_loader_xlm, device)
    
    # Save history
    xlm_history['train_loss'].append(train_loss)
    xlm_history['val_loss'].append(val_loss)
    xlm_history['val_accuracy'].append(val_accuracy)
    
    print(f"\nEpoch {epoch + 1} Results:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")
    print(f"  Val Acc:    {val_accuracy:.4f}")

print("\n✓ XLM-RoBERTa training completed!")

## Evaluate on Test Set

In [ ]:
# Create test datasets and loaders
test_dataset_bert = ReviewDataset(
    test_df[CONFIG['text_column']].tolist(),
    test_df['label'].tolist(),
    bert_tokenizer,
    CONFIG['max_length']
)
test_dataset_roberta = ReviewDataset(
    test_df[CONFIG['text_column']].tolist(),
    test_df['label'].tolist(),
    roberta_tokenizer,
    CONFIG['max_length']
)
test_dataset_xlm = ReviewDataset(
    test_df[CONFIG['text_column']].tolist(),
    test_df['label'].tolist(),
    xlm_tokenizer,
    CONFIG['max_length']
)

test_loader_bert = DataLoader(test_dataset_bert, batch_size=CONFIG['batch_size'])
test_loader_roberta = DataLoader(test_dataset_roberta, batch_size=CONFIG['batch_size'])
test_loader_xlm = DataLoader(test_dataset_xlm, batch_size=CONFIG['batch_size'])

print("Test datasets created")

In [ ]:
# Evaluate all models
print("="*70)
print("TEST SET EVALUATION")
print("="*70)

results = {}

for name, model, loader in [
    ('BERT', bert_model, test_loader_bert),
    ('RoBERTa', roberta_model, test_loader_roberta),
    ('XLM-RoBERTa', xlm_model, test_loader_xlm)
]:
    print(f"\nEvaluating {name}...")
    _, accuracy, predictions, true_labels = evaluate(model, loader, device)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, predictions, average='weighted', zero_division=0
    )
    
    results[name] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'predictions': predictions,
        'true_labels': true_labels
    }
    
    print(f"\n{name} Results:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")

print("\n" + "="*70)
print("✓ All models evaluated!")
print("="*70)

## Model Comparison

In [ ]:
# Create comparison table
comparison_data = []
for model_name, metrics in results.items():
    comparison_data.append({
        'Model': model_name,
        'Accuracy': metrics['accuracy'],
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'F1-Score': metrics['f1']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('F1-Score', ascending=False)

print("\n📊 MODEL COMPARISON TABLE")
print("="*70)
display(comparison_df.style.format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1-Score': '{:.4f}'
}).background_gradient(subset=['F1-Score'], cmap='YlGn'))

# Save to CSV
comparison_df.to_csv('model_comparison.csv', index=False)
print("\n✓ Saved to 'model_comparison.csv'")

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(14, 6))

metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(comparison_df))
width = 0.2

for i, metric in enumerate(metrics_to_plot):
    offset = width * (i - 1.5)
    ax.bar(x + offset, comparison_df[metric], width, label=metric, alpha=0.8)

ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Model'])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Chart saved as 'model_comparison.png'")

## Training History Visualization

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss curves
for name, history in [('BERT', bert_history), ('RoBERTa', roberta_history), ('XLM-RoBERTa', xlm_history)]:
    epochs = range(1, len(history['train_loss']) + 1)
    axes[0].plot(epochs, history['train_loss'], marker='o', label=f"{name} (Train)")
    axes[0].plot(epochs, history['val_loss'], marker='s', linestyle='--', label=f"{name} (Val)")

axes[0].set_xlabel('Epoch', fontweight='bold')
axes[0].set_ylabel('Loss', fontweight='bold')
axes[0].set_title('Training & Validation Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
for name, history in [('BERT', bert_history), ('RoBERTa', roberta_history), ('XLM-RoBERTa', xlm_history)]:
    epochs = range(1, len(history['val_accuracy']) + 1)
    axes[1].plot(epochs, history['val_accuracy'], marker='o', label=name)

axes[1].set_xlabel('Epoch', fontweight='bold')
axes[1].set_ylabel('Accuracy', fontweight='bold')
axes[1].set_title('Validation Accuracy', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Training history saved as 'training_history.png'")

## Confusion Matrices

In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
label_names = ['Negative', 'Neutral', 'Positive']

for idx, (name, metrics) in enumerate(results.items()):
    cm = confusion_matrix(metrics['true_labels'], metrics['predictions'])
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=label_names,
               yticklabels=label_names,
               ax=axes[idx],
               cbar_kws={'label': 'Count'})
    
    axes[idx].set_title(f'{name}\nAccuracy: {metrics["accuracy"]:.3f}', fontweight='bold')
    axes[idx].set_ylabel('True Label', fontweight='bold')
    axes[idx].set_xlabel('Predicted Label', fontweight='bold')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Confusion matrices saved as 'confusion_matrices.png'")

## Save Models

In [ ]:
import os

# Save all models
for name, model, tokenizer in [
    ('bert-base-uncased', bert_model, bert_tokenizer),
    ('roberta-base', roberta_model, roberta_tokenizer),
    ('xlm-roberta-base', xlm_model, xlm_tokenizer)
]:
    save_path = f'./models/{name}'
    os.makedirs(save_path, exist_ok=True)
    
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    
    print(f"✓ {name} saved to {save_path}")

print("\n✓ All models saved!")

---

## 🎉 Training Complete!

### Summary:
✅ Trained 3 transformer models (BERT, RoBERTa, XLM-RoBERTa)  
✅ Evaluated on test set  
✅ Generated comparison charts  
✅ Created confusion matrices  
✅ Saved trained models  

### Generated Files:
- `model_comparison.csv` - Performance metrics
- `model_comparison.png` - Comparison bar chart
- `training_history.png` - Training curves
- `confusion_matrices.png` - Confusion matrices
- `./models/*` - Saved model checkpoints

### Next Steps:
1. Use Part 3 notebook for predictions on new reviews
2. Analyze results for your report
3. Test models on different languages (if using European dataset)

---